# ⚙️ Notebook 07 – Model Optimization
**Goal:** improve the models by tuning **hyperparameters** (settings chosen *before* training) with cross-validated Grid Search.

**What we tune**
- TF-IDF: `ngram_range` (single words vs. word pairs) and `min_df`
- Logistic Regression: `C` (regularisation strength) and `class_weight`
- Naive Bayes: `alpha` (smoothing)

**Rules that avoid cheating**
- Tuning uses **only the training data** (5-fold cross-validation, scored by macro F1).
- The test set is touched **once**, at the very end.
- TF-IDF is inside a `Pipeline`, so it is re-fitted in every fold.

## 1. Import Libraries

In [ ]:
# Import tuning tools
import json
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

In [ ]:
# Set up project paths (works on your D: drive path, or falls back to the parent of 02_Notebooks)
from pathlib import Path

BASE_DIR = Path(r"D:\DataScience\notebooks\SentimentIQ")
if not BASE_DIR.exists():
    BASE_DIR = Path.cwd().parent          # fallback: notebook is running inside 02_Notebooks/

RAW_PATH       = BASE_DIR / "01_Data" / "Raw" / "Tweets.csv"
PROCESSED_DIR  = BASE_DIR / "01_Data" / "Processed"
CLEANED_PATH   = PROCESSED_DIR / "cleaned_tweets.csv"
MODELS_DIR     = BASE_DIR / "03_Models"
REPORTS_DIR    = BASE_DIR / "05_Reports"

for folder in (PROCESSED_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42      # same seed everywhere -> reproducible results
TEST_SIZE    = 0.20    # 80% training / 20% testing

print("Project folder :", BASE_DIR)
print("Raw data exists:", RAW_PATH.exists())

In [ ]:
# Load the cleaned dataset and recreate the SAME train/test split that was made in Notebook 4
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    df = pd.read_csv(CLEANED_PATH)
except FileNotFoundError:
    raise FileNotFoundError("cleaned_tweets.csv not found. Please run Notebook 03 first.")

df = df.dropna(subset=["cleaned_text", "airline_sentiment"]).reset_index(drop=True)

X = df["cleaned_text"]          # input feature
y = df["airline_sentiment"]     # target variable

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print("Train size:", len(X_train), "| Test size:", len(X_test))

## 2. Tune Logistic Regression

In [ ]:
# Pipeline = TF-IDF + Logistic Regression. 2 x 2 x 4 x 2 = 32 combinations x 5 folds = 160 fits.
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000, max_df=0.95, sublinear_tf=True)),
    ("clf",   LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

lr_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df":      [1, 2],
    "clf__C":             [0.5, 1, 5, 10],
    "clf__class_weight":  [None, "balanced"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
lr_search = GridSearchCV(lr_pipeline, lr_grid, cv=cv, scoring="f1_macro", n_jobs=-1, verbose=1)

start = time.time()
lr_search.fit(X_train, y_train)                       # test data is NOT used here
print(f"\n⏱ Finished in {time.time() - start:.0f} seconds")
print("Best CV macro F1 :", round(lr_search.best_score_, 4))
print("Best parameters  :", lr_search.best_params_)

In [ ]:
# Show the 10 best parameter combinations
lr_results = pd.DataFrame(lr_search.cv_results_)
cols = ["param_tfidf__ngram_range", "param_tfidf__min_df", "param_clf__C", "param_clf__class_weight",
        "mean_test_score", "std_test_score"]
lr_results.sort_values("mean_test_score", ascending=False)[cols].head(10).round(4).reset_index(drop=True)

In [ ]:
# How does the regularisation strength C change performance? (best score for each C value)
c_effect = lr_results.groupby("param_clf__C")["mean_test_score"].max()
plt.figure(figsize=(7, 4))
plt.plot(c_effect.index.astype(float), c_effect.values, marker="o", color="#3498db")
plt.xscale("log")
plt.xlabel("C (log scale)")
plt.ylabel("CV macro F1")
plt.title("Logistic Regression: effect of C")
plt.show()

## 3. Tune Multinomial Naive Bayes

In [ ]:
# Same idea for Naive Bayes: 2 x 2 x 4 = 16 combinations
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000, max_df=0.95, sublinear_tf=True)),
    ("clf",   MultinomialNB()),
])
nb_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df":      [1, 2],
    "clf__alpha":         [0.1, 0.3, 0.5, 1.0],
}
nb_search = GridSearchCV(nb_pipeline, nb_grid, cv=cv, scoring="f1_macro", n_jobs=-1, verbose=1)

start = time.time()
nb_search.fit(X_train, y_train)
print(f"\n⏱ Finished in {time.time() - start:.0f} seconds")
print("Best CV macro F1 :", round(nb_search.best_score_, 4))
print("Best parameters  :", nb_search.best_params_)

## 4. Compare Baseline vs Tuned Models on the Test Set

In [ ]:
# Evaluate the baseline Logistic Regression (Notebook 05 settings) and both tuned models on the test set
baseline_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
    ("clf",   LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced", random_state=RANDOM_STATE)),
]).fit(X_train, y_train)

candidates = {
    "Baseline Logistic Regression": baseline_pipe,
    "Tuned Logistic Regression":    lr_search.best_estimator_,
    "Tuned Naive Bayes":            nb_search.best_estimator_,
}

rows = []
for name, pipe in candidates.items():
    pred = pipe.predict(X_test)
    rows.append({"Model": name,
                 "Accuracy": accuracy_score(y_test, pred),
                 "F1 (macro)": f1_score(y_test, pred, average="macro"),
                 "F1 (weighted)": f1_score(y_test, pred, average="weighted")})
comparison = pd.DataFrame(rows).set_index("Model").round(4)
comparison

In [ ]:
# Visualise the comparison
comparison[["Accuracy", "F1 (macro)"]].plot(kind="bar", figsize=(9, 4.5), color=["#3498db", "#e67e22"], rot=15)
plt.ylim(0, 1)
plt.title("Baseline vs Tuned Models (Test Set)")
plt.ylabel("Score")
plt.show()

## 5. Select the Final Model

In [ ]:
# Choose the winner by CROSS-VALIDATION score (not by the test set, to avoid peeking)
if lr_search.best_score_ >= nb_search.best_score_:
    final_name, final_pipeline, final_search = "Logistic Regression", lr_search.best_estimator_, lr_search
else:
    final_name, final_pipeline, final_search = "Multinomial Naive Bayes", nb_search.best_estimator_, nb_search

print(f"🏆 Final model: {final_name}")
print("CV macro F1 :", round(final_search.best_score_, 4))
print("Parameters  :", final_search.best_params_)

final_pred = final_pipeline.predict(X_test)
print("\nTest accuracy :", round(accuracy_score(y_test, final_pred), 4))
print("Test macro F1 :", round(f1_score(y_test, final_pred, average="macro"), 4))
print()
print(classification_report(y_test, final_pred, digits=3))

In [ ]:
# Confusion matrix of the final model
labels = list(final_pipeline.named_steps["clf"].classes_)
cm = confusion_matrix(y_test, final_pred, labels=labels)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title(f"Final Model ({final_name}) – Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.savefig(REPORTS_DIR / "final_model_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Save the Optimized Pipeline and Tuning Results

In [ ]:
# Save the winning pipeline (TF-IDF + classifier) and the tuning results
try:
    joblib.dump(final_pipeline, MODELS_DIR / "optimized_pipeline.pkl")
    comparison.to_csv(REPORTS_DIR / "optimization_comparison.csv")
    lr_results.sort_values("mean_test_score", ascending=False).to_csv(REPORTS_DIR / "grid_search_lr_results.csv", index=False)

    best_info = {
        "final_model": final_name,
        "best_params": {k: str(v) for k, v in final_search.best_params_.items()},
        "cv_macro_f1": round(float(final_search.best_score_), 4),
        "test_accuracy": round(float(accuracy_score(y_test, final_pred)), 4),
        "test_macro_f1": round(float(f1_score(y_test, final_pred, average="macro")), 4),
    }
    with open(REPORTS_DIR / "best_model_info.json", "w") as f:
        json.dump(best_info, f, indent=2)
    print("✅ Saved optimized_pipeline.pkl and reports")
    print(json.dumps(best_info, indent=2))
except Exception as e:
    print("❌ Saving failed:", e)

## ✅ Summary
- Grid Search with 5-fold CV tuned both the TF-IDF settings and the classifier settings.
- The winner is chosen by cross-validation score; the test set is used only for the final report.
- Expect a modest gain (about +0.5 to +2 points of macro F1). Large jumps are not realistic for a bag-of-words model – and a huge jump would be a sign of leakage.

**Next:** Notebook 08 – Save the final model for the app and test it.